# Від прийому файлів переходимо до SILVER 

- Ідея полягає в тому, щоб створити SILVER таблиці, які будуть завантажуватися, коли в календарі позначена якась дата, як активний операційний день.
- До SILVER рівня відносяться:
1. Calendar - це глобальний рівень: Статус дня {W-робочий, H-Вихідний, O-операційний}
2. Журнал завантажень. По ньому визначаємо як виконується обробка даних
3. Таблиця платідних терміналів. ЇЇ копіюємо за terminals_d
4. Таблиця транзакцій. Переноситься з ev2_file_body, якщо в календары выдмычена дата того операцыйного дня.
5. Створюэмо траблицю клієнтів з таблиці транзакцій ev2_file_body  за полями: NAME, PASSPOR
6. Створюємо таблицю трназакцій з   ev2_file_body  і біля кожної транхакції ставимо CIFID клієнта.

В результаті цих дій ми отримаємо занальну "зіркову" структуру:
- Таблиця фактів:  ev2_silver_transcation
- Вимірювання: calendar
- Термінали: ev2_silver_terminal
- Клієнти: ev2_silver_customers

При цьому ще є протокольна таблиця розрахунків

## 1. Створення структури таблиць для SILVER LEVEL

## 1.1. Таблиця журналу розрахунків **psh_dwh_lh.dbo.ev2_silver_upload_grn**

In [2]:
print(f"Create Table psh_dwh_lh.dbo.ev2_silver_upload_grn")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_upload_grn")

spark.sql("""
  CREATE TABLE psh_dwh_lh.dbo.ev2_silver_upload_grn (
  upload_id       STRING,
  opdate          DATE,
  idt             TIMESTAMP,
  upload_status   STRING,
  upload_at      TIMESTAMP 
)
USING DELTA;     
"""
)

print(f"Table created")

StatementMeta(, 68a04820-42a8-4e8e-9705-850f24f18747, 11, Finished, Available, Finished, False)

Create Table psh_dwh_lh.dbo.ev2_silver_upload_grn
Table created


### 1.2. Таблиця платіжних терміналів **psh_dwh_lh.dbo.ev2_silver_terminals_dim**


In [1]:
print(f"Create Table psh_dwh_lh.dbo.ev2_silver_terminals_dim")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_terminals_dim")
spark.sql("""
    CREATE TABLE psh_dwh_lh.dbo.ev2_silver_terminals_dim (
            terminal_id   BIGINT,
            terminal_name STRING,
            isactive      STRING,
            upload_id     STRING,
            idt           TIMESTAMP
    )  USING DELTA;  
"""
)

print(f"Table created")

StatementMeta(, 810eccda-2900-459c-8d36-a7087f70c164, 3, Finished, Available, Finished, False)

Create Table psh_dwh_lh.dbo.ev2_silver_trerminals_dim
Table created


### 1.3. Таблиця клієнтів  **psh_dwh_lh.dbo.ev2_silver_customer_dim**

In [5]:
print(f"Create Table psh_dwh_lh.dbo.ev2_silver_customers_dim")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_customers_dim")

spark.sql("""
    CREATE TABLE psh_dwh_lh.dbo.ev2_silver_customers_dim (
            cust_id       STRING,
            cust_name     STRING,
            cust_doc      STRING,
            cust_status   STRING,
            upload_id     STRING,
            idt           TIMESTAMP
    )  USING DELTA;  
"""
)

print(f"Table created")

StatementMeta(, 68a04820-42a8-4e8e-9705-850f24f18747, 14, Finished, Available, Finished, False)

Create Table psh_dwh_lh.dbo.ev2_silver_customers_dim
Table created


### 1.4. Таблиця транзакцій  **psh_dwh_lh.dbo.ev2_silver_transactions**

In [6]:
print(f"psh_dwh_lh.dbo.ev2_silver_transactions")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_transactions")

spark.sql("""
    CREATE TABLE psh_dwh_lh.dbo.ev2_silver_transactions (
            tx_id         STRING,
            terminal_id   BIGINT,
            opdate        DATE,
            name          STRING,
            passport      STRING,
            amount        DECIMAL(18,2),
            charge        DECIMAL(18,2),
            status        STRING,
            file_name     STRING,
            file_id       STRING,
            upload_id     STRING,
            idt           TIMESTAMP
    )  USING DELTA;  
"""
)

print(f"Table created")

StatementMeta(, 68a04820-42a8-4e8e-9705-850f24f18747, 15, Finished, Available, Finished, False)

psh_dwh_lh.dbo.ev2_silver_transactions
Table created


## 2. Сервісні функції для відладки та розробки

### 2.1. Робота з календраем

In [1]:
# Статуси днів
import pandas as pd
df=spark.sql("""
   SELECT distinct A.STATUS
   FROM dbo.calendar A
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

StatementMeta(, 47846f9a-09cb-498b-9f0f-54250feb04f9, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d038f02-edf1-4bbd-8ad1-265439abc68e)

In [2]:
# вибрати мінмальну і максимальну дати
import pandas as pd
df=spark.sql("""
   SELECT  MIN(A.OPDATE) as midt, MAX(A.OPDATE) as madt 
   FROM dbo.calendar A
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

StatementMeta(, b12b34b5-a70e-4ee9-9795-0a3dd2525e54, 91, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 16b2114a-552b-4202-a08b-93808ee5037b)

In [3]:
# вибрати мінмальну і максимальну дати
import pandas as pd
df=spark.sql("""
   SELECT A.OPDATE, count(*)  as cntrec 
   FROM dbo.ev2_file_body A
   GROUP BY A.OPDATE
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

StatementMeta(, b12b34b5-a70e-4ee9-9795-0a3dd2525e54, 136, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0509000a-d550-4dfb-b6dd-3abf13660798)

### Встановити статус операційного дня в календарі

In [1]:
# Встановити статус операційного дня

pa_dt = "2025-07-12"
# Допусимі статуси: W-робочий, H-Вихіднийб O-Операційний відкритий, C-Операційний закритий
# Може бути тільки один операційний
pa_status = "O" 


def set_opdate_status(p_dt, p_status):
    """Встановити статус операційного дня"""
    # якщо ставимо статус "O" то всі інші треба закрити в "C"  
    if p_status=='O':
        spark.sql("""
                    UPDATE psh_dwh_lh.dbo.calendar
                    SET STATUS='C'
                    WHERE  /*OPDATE = :opdate and*/ STATUS=:status
                """, {"status": p_status})

    # Встановлюю статус поточного операційного дня
    spark.sql("""
                UPDATE psh_dwh_lh.dbo.calendar
                SET STATUS=:status
                WHERE  OPDATE = :opdate
            """, {"opdate": p_dt, "status": p_status})


# Запускаю функцію в роботу 
set_opdate_status(pa_dt, pa_status)
df_new_date=spark.sql(
    """
        SELECT C.OPDATE, C.STATUS FROM psh_dwh_lh.dbo.calendar C WHERE C.STATUS='O' 
        UNION ALL
        SELECT C.OPDATE, C.STATUS FROM psh_dwh_lh.dbo.calendar C WHERE C.STATUS='C' 
    """
)
df_new_date.show()



StatementMeta(, 536ba5c0-4693-4bb5-be09-35646bcabfce, 3, Finished, Available, Finished, False)

+----------+------+
|    OPDATE|STATUS|
+----------+------+
|2025-07-12|     O|
|2025-04-15|     C|
|2025-07-02|     C|
+----------+------+

